# Homework 3 - NYC taxi demand dataset

In [1]:
import polars as pl

DATA_DIR = "data"

MONEY_COLS = [
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "congestion_surcharge",
    "airport_fee",
]
VENDOR_IDS = [1, 2, 6, 7]
RATECODE_IDS = [1, 2, 3, 4, 5, 6, 99]

BOROUGH = pl.Enum(
    [
        "bronx",
        "brooklyn",
        "ewr",
        "manhattan",
        "queens",
        "staten_island",
        "unknown",
        "outside_nyc",
    ]
)
PAYMENT_TYPE = pl.Enum(["card", "cash", "other"])

## 1. Data reading

In [2]:
def read_rides() -> pl.LazyFrame:
    months = [
        pl.scan_parquet(f"{DATA_DIR}/{month}.parquet").with_columns(
            pl.col("tpep_pickup_datetime", "tpep_dropoff_datetime").dt.cast_time_unit(
                "us"
            )
        )
        for month in range(1, 13)
    ]
    return (
        pl.concat(months)
        .rename({"Airport_fee": "airport_fee"})
        .filter(
            pl.col("tpep_pickup_datetime").dt.year() == 2024,
            pl.col("tpep_dropoff_datetime") <= pl.datetime(2025, 1, 1),
        )
        .with_columns(
            pl.col("VendorID", "RatecodeID", "passenger_count", "payment_type").cast(
                pl.UInt8
            ),
            pl.col("PULocationID", "DOLocationID").cast(pl.UInt16),
            pl.col("store_and_fwd_flag").cast(pl.Enum(["N", "Y"])),
        )
    )


def read_zones() -> pl.LazyFrame:
    return pl.scan_csv(f"{DATA_DIR}/taxi_zone_lookup.csv").select(
        pl.col("LocationID").cast(pl.UInt16),
        pl.col("Borough")
        .replace("N/A", "outside_nyc")
        .str.to_lowercase()
        .str.replace(" ", "_")
        .cast(BOROUGH)
        .alias("borough"),
    )

In [3]:
read_rides().collect_schema()

Schema([('VendorID', UInt8),
        ('tpep_pickup_datetime', Datetime(time_unit='us', time_zone=None)),
        ('tpep_dropoff_datetime', Datetime(time_unit='us', time_zone=None)),
        ('passenger_count', UInt8),
        ('trip_distance', Float64),
        ('RatecodeID', UInt8),
        ('store_and_fwd_flag', Enum(categories=['N', 'Y'])),
        ('PULocationID', UInt16),
        ('DOLocationID', UInt16),
        ('payment_type', UInt8),
        ('fare_amount', Float64),
        ('extra', Float64),
        ('mta_tax', Float64),
        ('tip_amount', Float64),
        ('tolls_amount', Float64),
        ('improvement_surcharge', Float64),
        ('total_amount', Float64),
        ('congestion_surcharge', Float64),
        ('airport_fee', Float64)])

In [4]:
read_zones().collect()

LocationID,borough
u16,enum
1,"""ewr"""
2,"""queens"""
3,"""bronx"""
4,"""manhattan"""
5,"""staten_island"""
…,…
261,"""manhattan"""
262,"""manhattan"""
263,"""manhattan"""
